# Step 9 — SYSU-MM01 full-paper-like validation

This notebook runs SYSU-MM01 validation for the current main method:

**UPR-CRE v0.2 = prototype-guided relation refinement + confidence-filtered hard relation curriculum.**

It does not edit source code. Apply and push the Step 9 scripts first, then run this notebook.

Main purpose:

1. Prepare SYSU-MM01 on Kaggle.
2. Run paper-like Phase 1 if no checkpoint is provided.
3. Run Baseline vs UPR-CRE v0.2 Phase 2 in parallel on T4x2.
4. Save paper reference scores and local scores into CSV.
5. Archive and optionally push small results to GitHub.


## Block 0 — Configuration

Change only this block if needed. The default RegDB source matches your Kaggle dataset mount.

In [1]:
from pathlib import Path
from datetime import datetime

CFG = {
    # Repo
    "GITHUB_OWNER": "TranTruongMMCII",
    "REPO_NAME": "UIT.CS2309",
    "BRANCH": "feature/upr-cre",
    "WORK_DIR": "/kaggle/working",

    # SYSU dataset. Empty SYSU_SOURCE = auto-detect under /kaggle/input.
    "DATA_ROOT": "/kaggle/working/VIREID_Dataset",
    "SYSU_SOURCE": "/kaggle/input/datasets/coconutjean/sysumm01",

    # Optional Phase-1 checkpoint. Leave empty to train Phase 1.
    # Example: /kaggle/input/<your-sysu-checkpoint-dataset>/model_20.pth
    "PHASE1_CKPT_PATH": "/kaggle/input/datasets/truongtranmmcii/uit-cs2309-checkpoint/model_20.pth",

    # Run mode. "paper" = official-like SYSU config. "short" = fast sanity check.
    # "RUN_MODE": "paper",
    "RUN_MODE": "short",

    # Paper-like config. Official script uses stage1=20, stage2 default 120, lr=0.0003, milestones=30 70.
    "PAPER_STAGE1_EPOCH": 20,
    "PAPER_STAGE2_EPOCH": 120,
    "PAPER_LR": 0.0003,
    "PAPER_MILESTONES": "30 70",
    "PAPER_BATCH_PIDNUM": 8,
    "PAPER_PID_NUMSAMPLE": 4,
    "PAPER_TEST_BATCH": 128,
    "PAPER_NUM_WORKERS": 4,

    # Short config for debugging only.
    "SHORT_STAGE1_EPOCH": 5,
    "SHORT_STAGE2_EPOCH": 15,
    "SHORT_MILESTONES": "8 12",

    "SEED": 1,
    "DEVICE_A": 0,
    "DEVICE_B": 1,

    # UPR-CRE v0.2 best config from RegDB development.
    "UPR_BETA": 0.1,
    "UPR_GAMMA": 0.0,
    "UPR_WARMUP_EPOCH": 2,
    "UPR_FILTER_START_EPOCH": 2,
    "UPR_FILTER_END_EPOCH": 10,
    "UPR_FILTER_START_RATIO": 0.55,
    "UPR_FILTER_END_RATIO": 1.0,
    "UPR_FILTER_MIN_PAIRS": 40,

    # Relation diagnostics every N epochs to reduce file count in long paper runs.
    "RELATION_STATS_EVERY": 5,

    # Backup to GitHub using normal git push. Requires Kaggle secret GITHUB_TOKEN.
    "PUSH_SMALL_BACKUP_TO_GIT": True,
    "GITHUB_TOKEN_SECRET": "GITHUB_TOKEN",
    "GIT_USER_NAME": "Kaggle Bot",
    "GIT_USER_EMAIL": "kaggle-bot@example.com",
}

if CFG["RUN_MODE"] == "paper":
    CFG["STAGE1_EPOCH"] = CFG["PAPER_STAGE1_EPOCH"]
    CFG["STAGE2_EPOCH"] = CFG["PAPER_STAGE2_EPOCH"]
    CFG["LR"] = CFG["PAPER_LR"]
    CFG["MILESTONES"] = CFG["PAPER_MILESTONES"]
else:
    CFG["STAGE1_EPOCH"] = CFG["SHORT_STAGE1_EPOCH"]
    CFG["STAGE2_EPOCH"] = CFG["SHORT_STAGE2_EPOCH"]
    CFG["LR"] = CFG["PAPER_LR"]
    CFG["MILESTONES"] = CFG["SHORT_MILESTONES"]

CFG["BATCH_PIDNUM"] = CFG["PAPER_BATCH_PIDNUM"]
CFG["PID_NUMSAMPLE"] = CFG["PAPER_PID_NUMSAMPLE"]
CFG["TEST_BATCH"] = CFG["PAPER_TEST_BATCH"]
CFG["NUM_WORKERS"] = CFG["PAPER_NUM_WORKERS"]

RUN_SUFFIX = datetime.utcnow().strftime(f"sysu_{CFG['RUN_MODE']}_%Y%m%d_%H%M%S")
print("RUN_SUFFIX =", RUN_SUFFIX)
print("CFG stage1/stage2 =", CFG["STAGE1_EPOCH"], CFG["STAGE2_EPOCH"])
repo_dir = Path(CFG["WORK_DIR"]) / CFG["REPO_NAME"]
wsl_dir = repo_dir / "WSL_ReID"
print("repo_dir =", repo_dir)
print("wsl_dir =", wsl_dir)


RUN_SUFFIX = sysu_short_20260629_130838
CFG stage1/stage2 = 5 15
repo_dir = /kaggle/working/UIT.CS2309
wsl_dir = /kaggle/working/UIT.CS2309/WSL_ReID


/tmp/ipykernel_23/3336559354.py:78: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  RUN_SUFFIX = datetime.utcnow().strftime(f"sysu_{CFG['RUN_MODE']}_%Y%m%d_%H%M%S")


## Block 1 — Pull repo and checkout branch

This block resets the Kaggle copy to `origin/feature/upr-cre`. It does not change code.

In [2]:
import subprocess, os
from pathlib import Path

repo_dir = Path(CFG["WORK_DIR"]) / CFG["REPO_NAME"]
repo_url = f"https://github.com/{CFG['GITHUB_OWNER']}/{CFG['REPO_NAME']}.git"

if not repo_dir.exists():
    subprocess.run(["git", "clone", "--branch", CFG["BRANCH"], repo_url, str(repo_dir)], check=True)
else:
    subprocess.run(["git", "fetch", "origin", CFG["BRANCH"]], cwd=repo_dir, check=True)
    subprocess.run(["git", "checkout", CFG["BRANCH"]], cwd=repo_dir, check=True)
    subprocess.run(["git", "reset", "--hard", f"origin/{CFG['BRANCH']}"], cwd=repo_dir, check=True)

wsl_dir = repo_dir / "WSL_ReID"
commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=repo_dir).decode().strip()
print("Commit:", commit)

required = [
    "scripts/prepare_sysu_kaggle.py",
    "scripts/check_sysu_env.py",
    "scripts/run_sysu_phase1_paper.sh",
    "scripts/run_sysu_phase2_baseline_paper.sh",
    "scripts/run_sysu_phase2_upr_v02_paper.sh",
    "main.py",
    "wsl.py",
    "relation_metrics.py",
]
missing = [p for p in required if not (wsl_dir / p).exists()]
if missing:
    raise FileNotFoundError("Missing Step 9 files: " + ", ".join(missing))

for rel in ["scripts/prepare_sysu_kaggle.py", "scripts/check_sysu_env.py", "main.py", "wsl.py"]:
    subprocess.run(["python", "-m", "py_compile", str(wsl_dir / rel)], check=True)
for rel in ["scripts/run_sysu_phase1_paper.sh", "scripts/run_sysu_phase2_baseline_paper.sh", "scripts/run_sysu_phase2_upr_v02_paper.sh"]:
    subprocess.run(["bash", "-n", str(wsl_dir / rel)], check=True)
print("Step 9 source checks OK.")


Cloning into '/kaggle/working/UIT.CS2309'...


Commit: 5cb2e45
Step 9 source checks OK.


## Block 2 — Install requirements, apply compatibility patches, prepare SYSU-MM01

In [3]:
import subprocess, os
from pathlib import Path

wsl_dir = Path(CFG["WORK_DIR"]) / CFG["REPO_NAME"] / "WSL_ReID"

subprocess.run(["pip", "install", "-q", "-r", "requirements-kaggle.txt"], cwd=wsl_dir, check=True)
if (wsl_dir / "scripts/apply_kaggle_compat_patches.py").exists():
    subprocess.run(["python", "scripts/apply_kaggle_compat_patches.py"], cwd=wsl_dir, check=True)

prepare_cmd = ["python", "scripts/prepare_sysu_kaggle.py", "--data-root", CFG["DATA_ROOT"]]
if CFG["SYSU_SOURCE"]:
    prepare_cmd += ["--sysu-source", CFG["SYSU_SOURCE"]]
print("Prepare command:", " ".join(prepare_cmd))
subprocess.run(prepare_cmd, cwd=wsl_dir, check=True)
subprocess.run(["python", "scripts/check_sysu_env.py", "--data-root", CFG["DATA_ROOT"]], cwd=wsl_dir, check=True)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.6 MB/s eta 0:00:00
No compatibility patches were needed.
Prepare command: python scripts/prepare_sysu_kaggle.py --data-root /kaggle/working/VIREID_Dataset --sysu-source /kaggle/input/datasets/coconutjean/sysumm01
SYSU source: /kaggle/input/datasets/coconutjean/sysumm01
SYSU destination: /kaggle/working/VIREID_Dataset/SYSU-MM01
SYSU train+val identities: 395
SYSU RGB train images: 22258
SYSU IR train images: 11909


Processing SYSU ir images: 100%|██████████| 11909/11909 [00:51<00:00, 230.51it/s]


Created SYSU preprocessed numpy files.
SYSU prepared at: /kaggle/working/VIREID_Dataset/SYSU-MM01
train_rgb_info: (22258, 3) train_rgb_images: (22258, 288, 144, 3)
train_ir_info: (11909, 3) train_ir_images: (11909, 288, 144, 3)
Use training argument: --data-path /kaggle/working/VIREID_Dataset
=== PyTorch / GPU ===
torch: 2.10.0+cu128
cuda available: True
gpu count: 2
gpu 0: Tesla T4
gpu 1: Tesla T4

=== SYSU-MM01 layout ===
/kaggle/working/VIREID_Dataset/SYSU-MM01/cam1 OK
/kaggle/working/VIREID_Dataset/SYSU-MM01/cam2 OK
/kaggle/working/VIREID_Dataset/SYSU-MM01/cam3 OK
/kaggle/working/VIREID_Dataset/SYSU-MM01/cam4 OK
/kaggle/working/VIREID_Dataset/SYSU-MM01/cam5 OK
/kaggle/working/VIREID_Dataset/SYSU-MM01/cam6 OK
/kaggle/working/VIREID_Dataset/SYSU-MM01/exp OK
/kaggle/working/VIREID_Dataset/SYSU-MM01/train_rgb_modified_img.npy OK
/kaggle/working/VIREID_Dataset/SYSU-MM01/train_rgb_info.npy OK
/kaggle/working/VIREID_Dataset/SYSU-MM01/train_ir_modified_img.npy OK
/kaggle/working/VIREID_Dat

CompletedProcess(args=['python', 'scripts/check_sysu_env.py', '--data-root', '/kaggle/working/VIREID_Dataset'], returncode=0)

## Block 3 — Save official paper reference scores

In [4]:
import csv
from pathlib import Path

# Values are percentages from the ICCV 2025 paper tables.
paper_rows = [
    {"source": "paper_main", "method": "WSVIReID", "setting": "all_search", "rank1": 70.4, "rank10": 95.8, "rank20": 98.8, "map": 66.6, "minp": 52.7},
    {"source": "paper_main", "method": "WSVIReID", "setting": "indoor_search", "rank1": 76.5, "rank10": 97.5, "rank20": 99.3, "map": 80.2, "minp": 76.6},
    {"source": "paper_ablation", "method": "B", "setting": "all_search", "rank1": 47.8, "rank10": "", "rank20": "", "map": 47.2, "minp": ""},
    {"source": "paper_ablation", "method": "B+CMCL_no_CRE", "setting": "all_search", "rank1": 66.7, "rank10": "", "rank20": "", "map": 62.8, "minp": ""},
    {"source": "paper_ablation", "method": "B+CRE+CMCL", "setting": "all_search", "rank1": 68.0, "rank10": "", "rank20": "", "map": 64.6, "minp": ""},
    {"source": "paper_ablation", "method": "B+CRE+CMCL+CLAE", "setting": "all_search", "rank1": 70.4, "rank10": "", "rank20": "", "map": 66.6, "minp": ""},
]
ref_path = Path("/kaggle/working/sysu_paper_reference_scores.csv")
with ref_path.open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(paper_rows[0].keys()))
    writer.writeheader()
    writer.writerows(paper_rows)
print(ref_path.read_text())


source,method,setting,rank1,rank10,rank20,map,minp
paper_main,WSVIReID,all_search,70.4,95.8,98.8,66.6,52.7
paper_main,WSVIReID,indoor_search,76.5,97.5,99.3,80.2,76.6
paper_ablation,B,all_search,47.8,,,47.2,
paper_ablation,B+CMCL_no_CRE,all_search,66.7,,,62.8,
paper_ablation,B+CRE+CMCL,all_search,68.0,,,64.6,
paper_ablation,B+CRE+CMCL+CLAE,all_search,70.4,,,66.6,



## Block 4 — Train or locate Phase-1 checkpoint

In [5]:
%%time
from pathlib import Path
import subprocess, os, json, shutil

wsl_dir = Path(CFG["WORK_DIR"]) / CFG["REPO_NAME"] / "WSL_ReID"
phase1_ckpt = Path(CFG["PHASE1_CKPT_PATH"]) if CFG["PHASE1_CKPT_PATH"] else None

if phase1_ckpt is not None and phase1_ckpt.exists():
    print("Using provided Phase-1 checkpoint:", phase1_ckpt)
else:
    run_name = f"sysu_phase1_s{CFG['STAGE1_EPOCH']}_{RUN_SUFFIX}"
    log_path = Path("/kaggle/working/run_logs") / f"{run_name}.log"
    log_path.parent.mkdir(parents=True, exist_ok=True)
    env = os.environ.copy()
    env.update({
        "DATA_ROOT": CFG["DATA_ROOT"],
        "SYSU_SOURCE": CFG["SYSU_SOURCE"],
        "RUN_NAME": run_name,
        "DEVICE": str(CFG["DEVICE_A"]),
        "SEED": str(CFG["SEED"]),
        "STAGE1_EPOCH": str(CFG["STAGE1_EPOCH"]),
        "LR": str(CFG["LR"]),
        "MILESTONES": CFG["MILESTONES"],
        "BATCH_PIDNUM": str(CFG["BATCH_PIDNUM"]),
        "PID_NUMSAMPLE": str(CFG["PID_NUMSAMPLE"]),
        "TEST_BATCH": str(CFG["TEST_BATCH"]),
        "NUM_WORKERS": str(CFG["NUM_WORKERS"]),
    })
    print("Training Phase-1. Log:", log_path)
    with log_path.open("w") as f:
        subprocess.run(["bash", "scripts/run_sysu_phase1_paper.sh"], cwd=wsl_dir, env=env, stdout=f, stderr=subprocess.STDOUT, check=True)
    print(log_path.read_text()[-4000:])

    # Locate checkpoint. SYSU save path has no _trial suffix.
    candidates = sorted((wsl_dir.parent).rglob(f"model_{CFG['STAGE1_EPOCH']}.pth"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not candidates:
        raise FileNotFoundError(f"Could not find model_{CFG['STAGE1_EPOCH']}.pth after Phase-1")
    phase1_ckpt = candidates[0]

print("PHASE1_CKPT =", phase1_ckpt)
print("exists =", phase1_ckpt.exists(), "size MB =", phase1_ckpt.stat().st_size / (1024*1024))
Path("/kaggle/working/sysu_phase1_ckpt_path.txt").write_text(str(phase1_ckpt))


Using provided Phase-1 checkpoint: /kaggle/input/datasets/truongtranmmcii/uit-cs2309-checkpoint/model_20.pth
PHASE1_CKPT = /kaggle/input/datasets/truongtranmmcii/uit-cs2309-checkpoint/model_20.pth
exists = True size MB = 99.31389617919922
CPU times: user 990 µs, sys: 90 µs, total: 1.08 ms
Wall time: 6.26 ms


73

## Block 5 — Start baseline and UPR-CRE v0.2 Phase-2 runs in background on T4x2


In [6]:
from pathlib import Path
import subprocess, os, json

wsl_dir = Path(CFG["WORK_DIR"]) / CFG["REPO_NAME"] / "WSL_ReID"
phase1_ckpt = Path("/kaggle/working/sysu_phase1_ckpt_path.txt").read_text().strip()
commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=wsl_dir).decode().strip()
run_suffix = f"sysu_{CFG['RUN_MODE']}_{RUN_SUFFIX}_{commit}"
Path("/kaggle/working/sysu_run_suffix.txt").write_text(run_suffix)

run_logs_dir = Path("/kaggle/working/run_logs")
run_logs_dir.mkdir(parents=True, exist_ok=True)
pids_dir = Path("/kaggle/working/pids")
pids_dir.mkdir(parents=True, exist_ok=True)

def start_job(tag, enable_upr, gpu):
    save_path = f"{tag}_{run_suffix}"
    relation_dir = f"../saved_sysu_resnet/{save_path}/relation_stats"
    cmd = [
        "python", "-u", "main.py",
        "--dataset", "sysu",
        "--data-path", CFG["DATA_ROOT"],
        "--debug", "wsl",
        "--save-path", save_path,
        "--arch", "resnet",
        "--seed", str(CFG["SEED"]),
        "--model-path", phase1_ckpt,
        "--stage1-epoch", "0",
        "--stage2-epoch", str(CFG["STAGE2_EPOCH"]),
        "--lr", str(CFG["LR"]),
        "--batch-pidnum", str(CFG["BATCH_PIDNUM"]),
        "--pid-numsample", str(CFG["PID_NUMSAMPLE"]),
        "--test-batch", str(CFG["TEST_BATCH"]),
        "--num-workers", str(CFG["NUM_WORKERS"]),
        "--device", "0",
        "--milestones", *CFG["MILESTONES"].split(),
        "--search-mode", "all",
        "--gall-mode", "single",
        "--save-relation-stats",
        "--relation-stats-every", str(CFG["RELATION_STATS_EVERY"]),
        "--relation-stats-dir", relation_dir,
    ]
    if enable_upr:
        cmd += [
            "--upr-cre",
            "--upr-beta", str(CFG["UPR_BETA"]),
            "--upr-gamma", str(CFG["UPR_GAMMA"]),
            "--upr-margin-weight", "1.0",
            "--upr-warmup-epoch", str(CFG["UPR_WARMUP_EPOCH"]),
            "--upr-filter",
            "--upr-filter-start-epoch", str(CFG["UPR_FILTER_START_EPOCH"]),
            "--upr-filter-end-epoch", str(CFG["UPR_FILTER_END_EPOCH"]),
            "--upr-filter-start-ratio", str(CFG["UPR_FILTER_START_RATIO"]),
            "--upr-filter-end-ratio", str(CFG["UPR_FILTER_END_RATIO"]),
            "--upr-filter-min-pairs", str(CFG["UPR_FILTER_MIN_PAIRS"]),
        ]
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu)
    log_path = run_logs_dir / f"{save_path}.log"
    print("Starting", tag, "on GPU", gpu)
    print("Log:", log_path)
    print("Command:", " ".join(cmd))
    f = log_path.open("w")
    proc = subprocess.Popen(cmd, cwd=wsl_dir, env=env, stdout=f, stderr=subprocess.STDOUT)
    (pids_dir / f"{tag}.pid").write_text(str(proc.pid))
    return {"tag": tag, "enable_upr": enable_upr, "gpu": gpu, "pid": proc.pid, "save_path": save_path, "log_path": str(log_path), "relation_stats_dir": relation_dir}

jobs = [
    start_job("baseline_sysu_p2s" + str(CFG["STAGE2_EPOCH"]), False, CFG["DEVICE_A"]),
    start_job("upr_v02_sysu_p2s" + str(CFG["STAGE2_EPOCH"]), True, CFG["DEVICE_B"]),
]
info = {"run_suffix": run_suffix, "commit": commit, "phase1_ckpt": phase1_ckpt, "cfg": CFG, "jobs": jobs}
Path("/kaggle/working/sysu_runtime_info.json").write_text(json.dumps(info, indent=2))
print(json.dumps(jobs, indent=2))


Starting baseline_sysu_p2s15 on GPU 0
Log: /kaggle/working/run_logs/baseline_sysu_p2s15_sysu_short_sysu_short_20260629_130838_5cb2e45.log
Command: python -u main.py --dataset sysu --data-path /kaggle/working/VIREID_Dataset --debug wsl --save-path baseline_sysu_p2s15_sysu_short_sysu_short_20260629_130838_5cb2e45 --arch resnet --seed 1 --model-path /kaggle/input/datasets/truongtranmmcii/uit-cs2309-checkpoint/model_20.pth --stage1-epoch 0 --stage2-epoch 15 --lr 0.0003 --batch-pidnum 8 --pid-numsample 4 --test-batch 128 --num-workers 4 --device 0 --milestones 8 12 --search-mode all --gall-mode single --save-relation-stats --relation-stats-every 5 --relation-stats-dir ../saved_sysu_resnet/baseline_sysu_p2s15_sysu_short_sysu_short_20260629_130838_5cb2e45/relation_stats
Starting upr_v02_sysu_p2s15 on GPU 1
Log: /kaggle/working/run_logs/upr_v02_sysu_p2s15_sysu_short_sysu_short_20260629_130838_5cb2e45.log
Command: python -u main.py --dataset sysu --data-path /kaggle/working/VIREID_Dataset --deb

## Block 6 — Monitor running jobs


In [7]:
import subprocess, json
from pathlib import Path

info = json.loads(Path("/kaggle/working/sysu_runtime_info.json").read_text())
subprocess.run(["nvidia-smi"])
for job in info["jobs"]:
    print("\n=====", job["tag"], "=====")
    subprocess.run(["tail", "-n", "60", job["log_path"]])


Mon Jun 29 13:11:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
%%time
import time, json, subprocess
from pathlib import Path

info = json.loads(Path("/kaggle/working/sysu_runtime_info.json").read_text())
pids = [int(job["pid"]) for job in info["jobs"]]
print("Tracking PIDs:", pids)

while True:
    running = []
    for pid in pids:
        ret = subprocess.run(["bash", "-lc", f"kill -0 {pid} 2>/dev/null"], check=False)
        if ret.returncode == 0:
            running.append(pid)
    if not running:
        print("All jobs finished.")
        break
    print("Still running:", running)
    subprocess.run(["nvidia-smi", "--query-compute-apps=pid,used_memory", "--format=csv,noheader"], check=False)
    time.sleep(120)


Tracking PIDs: [87, 88]
Still running: [87, 88]
Still running: [87, 88]
87, 2112 MiB
88, 2112 MiB
Still running: [87, 88]
87, 10246 MiB
88, 10246 MiB
Still running: [87, 88]
87, 10246 MiB
88, 10246 MiB
Still running: [87, 88]
87, 10246 MiB
88, 10246 MiB
Still running: [87, 88]
87, 10246 MiB
88, 10246 MiB
Still running: [87, 88]
87, 10246 MiB
88, 10246 MiB
Still running: [87, 88]
87, 10246 MiB
88, 10246 MiB
Still running: [87, 88]
87, 10246 MiB
88, 10246 MiB
Still running: [87, 88]
87, 10246 MiB
88, 10246 MiB
Still running: [87, 88]
87, 10246 MiB
88, 10246 MiB
Still running: [87, 88]
87, 10246 MiB
88, 10246 MiB
Still running: [87, 88]
87, 10246 MiB
88, 10246 MiB
Still running: [87, 88]
87, 10246 MiB
88, 10246 MiB
Still running: [87, 88]
87, 10246 MiB
88, 10246 MiB
Still running: [87, 88]
87, 10246 MiB
88, 10246 MiB
Still running: [87, 88]
87, 10246 MiB
88, 10246 MiB
Still running: [87, 88]
87, 10246 MiB
88, 10246 MiB
Still running: [87, 88]
87, 10246 MiB
88, 10246 MiB
Still running: [87

## Block 7 — Collect local scores and create comparison CSV


In [9]:
import csv, json, re, subprocess
from pathlib import Path

wsl_dir = Path(CFG["WORK_DIR"]) / CFG["REPO_NAME"] / "WSL_ReID"
info = json.loads(Path("/kaggle/working/sysu_runtime_info.json").read_text())

def parse_best_metrics(log_text):
    matches = re.findall(r"R1:([0-9.]+);\s+R10:([0-9.]+);\s+R20:([0-9.]+);\s+mAP:([0-9.]+);\s+mINP:([0-9.]+)", log_text)
    if not matches:
        return None
    # Values in logs are fractions. Convert to percent for comparison with paper.
    best = max(matches, key=lambda x: float(x[3]))
    r1, r10, r20, map_, minp = [float(x) * 100.0 for x in best]
    return r1, r10, r20, map_, minp

rows = []
# Paper reference rows.
with Path("/kaggle/working/sysu_paper_reference_scores.csv").open() as f:
    for row in csv.DictReader(f):
        if row["source"] == "paper_main":
            rows.append({
                "source": row["source"],
                "method": row["method"],
                "setting": row["setting"],
                "run_mode": "paper_reported",
                "rank1_percent": row["rank1"],
                "rank10_percent": row["rank10"],
                "rank20_percent": row["rank20"],
                "map_percent": row["map"],
                "minp_percent": row["minp"],
                "common_accuracy": "",
                "mutual_match_ratio": "",
                "save_path": "",
            })

for job in info["jobs"]:
    run_dir = wsl_dir.parent / "saved_sysu_resnet" / job["save_path"]
    log_path = run_dir / "log" / "log.txt"
    stats_dir = run_dir / "relation_stats"
    stats_csv = stats_dir / "relation_stats_summary.csv"
    if stats_dir.exists() and any(stats_dir.glob("epoch_*.json")):
        subprocess.run([
            "python", "scripts/collect_relation_stats.py",
            "--stats-dir", str(stats_dir),
            "--csv-output", str(stats_csv),
        ], cwd=wsl_dir, check=True)
    metrics = parse_best_metrics(log_path.read_text() if log_path.exists() else "")
    last_stats = {}
    if stats_csv.exists():
        with stats_csv.open() as f:
            stats_rows = list(csv.DictReader(f))
            if stats_rows:
                last_stats = stats_rows[-1]
    rows.append({
        "source": "local",
        "method": "UPR-CRE-v02" if job["enable_upr"] else "local_baseline_WSL",
        "setting": "all_search",
        "run_mode": CFG["RUN_MODE"],
        "rank1_percent": f"{metrics[0]:.4f}" if metrics else "",
        "rank10_percent": f"{metrics[1]:.4f}" if metrics else "",
        "rank20_percent": f"{metrics[2]:.4f}" if metrics else "",
        "map_percent": f"{metrics[3]:.4f}" if metrics else "",
        "minp_percent": f"{metrics[4]:.4f}" if metrics else "",
        "common_accuracy": last_stats.get("common_accuracy", ""),
        "mutual_match_ratio": last_stats.get("mutual_match_ratio", ""),
        "save_path": job["save_path"],
    })

out = Path("/kaggle/working/sysu_fullpaper_baseline_vs_upr_summary.csv")
with out.open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)
print(out.read_text())


/kaggle/working/UIT.CS2309/saved_sysu_resnet/baseline_sysu_p2s15_sysu_short_sysu_short_20260629_130838_5cb2e45/relation_stats/relation_stats_summary.csv
/kaggle/working/UIT.CS2309/saved_sysu_resnet/upr_v02_sysu_p2s15_sysu_short_sysu_short_20260629_130838_5cb2e45/relation_stats/relation_stats_summary.csv
source,method,setting,run_mode,rank1_percent,rank10_percent,rank20_percent,map_percent,minp_percent,common_accuracy,mutual_match_ratio,save_path
paper_main,WSVIReID,all_search,paper_reported,70.4,95.8,98.8,66.6,52.7,,,
paper_main,WSVIReID,indoor_search,paper_reported,76.5,97.5,99.3,80.2,76.6,,,
local,local_baseline_WSL,all_search,short,57.8300,91.3100,95.9100,55.2200,40.7100,0.956386292834891,0.8746594005449592,baseline_sysu_p2s15_sysu_short_sysu_short_20260629_130838_5cb2e45
local,UPR-CRE-v02,all_search,short,60.1300,92.6900,97.0400,57.3300,42.6000,0.9633333333333334,0.8174386920980926,upr_v02_sysu_p2s15_sysu_short_sysu_short_20260629_130838_5cb2e45



## Block 8 — Archive outputs


In [10]:
from pathlib import Path
import json, shutil, subprocess

info = json.loads(Path("/kaggle/working/sysu_runtime_info.json").read_text())
wsl_dir = Path(CFG["WORK_DIR"]) / CFG["REPO_NAME"] / "WSL_ReID"
artifact_dir = Path("/kaggle/working/artifacts_step9_sysu_fullpaper")
if artifact_dir.exists():
    shutil.rmtree(artifact_dir)
(artifact_dir / "logs").mkdir(parents=True, exist_ok=True)
(artifact_dir / "relation_stats").mkdir(parents=True, exist_ok=True)

for p in [
    Path("/kaggle/working/sysu_runtime_info.json"),
    Path("/kaggle/working/sysu_paper_reference_scores.csv"),
    Path("/kaggle/working/sysu_fullpaper_baseline_vs_upr_summary.csv"),
    Path("/kaggle/working/sysu_phase1_ckpt_path.txt"),
]:
    if p.exists():
        shutil.copy2(p, artifact_dir / p.name)

for job in info["jobs"]:
    run_dir = wsl_dir.parent / "saved_sysu_resnet" / job["save_path"]
    log = run_dir / "log" / "log.txt"
    if log.exists():
        shutil.copy2(log, artifact_dir / "logs" / f"{job['save_path']}_log.txt")
    stats = run_dir / "relation_stats" / "relation_stats_summary.csv"
    if stats.exists():
        shutil.copy2(stats, artifact_dir / "relation_stats" / f"{job['save_path']}_relation_stats_summary.csv")

# Phase1 log if present.
for p in Path("/kaggle/working/run_logs").glob("sysu_phase1*.log"):
    shutil.copy2(p, artifact_dir / "logs" / p.name)

tar_path = Path("/kaggle/working/artifacts_step9_sysu_fullpaper.tar.gz")
if tar_path.exists():
    tar_path.unlink()
subprocess.run(["tar", "-czf", str(tar_path), "-C", "/kaggle/working", artifact_dir.name], check=True)
print("Artifact:", tar_path)
print("Size MB:", tar_path.stat().st_size / (1024*1024))


Artifact: /kaggle/working/artifacts_step9_sysu_fullpaper.tar.gz
Size MB: 0.006028175354003906


## Block 9 — Optional: push small backup to GitHub using old git push method


In [11]:
from pathlib import Path
import os, json, shutil, subprocess, textwrap, hashlib

if not CFG["PUSH_SMALL_BACKUP_TO_GIT"]:
    print("Git backup disabled.")
else:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret(CFG["GITHUB_TOKEN_SECRET"]).strip()
    repo_dir = Path(CFG["WORK_DIR"]) / CFG["REPO_NAME"]
    artifact_dir = Path("/kaggle/working/artifacts_step9_sysu_fullpaper")
    tar_path = Path("/kaggle/working/artifacts_step9_sysu_fullpaper.tar.gz")
    info = json.loads(Path("/kaggle/working/sysu_runtime_info.json").read_text())
    run_id = info["run_suffix"]
    backup_dir = repo_dir / "experiments" / "kaggle_runs" / run_id
    if backup_dir.exists():
        shutil.rmtree(backup_dir)
    backup_dir.mkdir(parents=True, exist_ok=True)
    if artifact_dir.exists():
        for item in artifact_dir.iterdir():
            dst = backup_dir / item.name
            if item.is_dir():
                shutil.copytree(item, dst, dirs_exist_ok=True)
            else:
                shutil.copy2(item, dst)
    manifest = {
        "run_id": run_id,
        "branch": CFG["BRANCH"],
        "artifact_tar": str(tar_path) if tar_path.exists() else None,
        "note": "Large tar/checkpoints are not committed. Upload to Kaggle Dataset/Input if needed.",
    }
    (backup_dir / "manifest.json").write_text(json.dumps(manifest, indent=2))
    (backup_dir / "README.md").write_text(f"# SYSU Step 9 backup: {run_id}\n\nSmall logs/CSV summary from Kaggle. Large artifacts are not committed.\n")

    askpass = Path(CFG["WORK_DIR"]) / "git_askpass_sysu_push.sh"
    askpass.write_text(textwrap.dedent(f"""\
#!/bin/sh
case "$1" in
  *Username*) echo "{CFG['GITHUB_OWNER']}" ;;
  *Password*) echo "{token}" ;;
  *) echo "" ;;
esac
"""))
    askpass.chmod(0o700)
    env = os.environ.copy()
    env["GIT_ASKPASS"] = str(askpass)
    env["GIT_TERMINAL_PROMPT"] = "0"
    try:
        subprocess.run(["git", "config", "user.name", CFG.get("GIT_USER_NAME", "Kaggle Bot")], cwd=repo_dir, env=env, check=True)
        subprocess.run(["git", "config", "user.email", CFG.get("GIT_USER_EMAIL", "kaggle-bot@example.com")], cwd=repo_dir, env=env, check=True)
        subprocess.run(["git", "pull", "--rebase", "origin", CFG["BRANCH"]], cwd=repo_dir, env=env, check=True)
        subprocess.run(["git", "add", "-f", str(backup_dir.relative_to(repo_dir))], cwd=repo_dir, env=env, check=True)
        diff = subprocess.run(["git", "diff", "--cached", "--quiet"], cwd=repo_dir, env=env)
        if diff.returncode != 0:
            subprocess.run(["git", "commit", "-m", f"exp: add SYSU Step 9 backup {run_id}"], cwd=repo_dir, env=env, check=True)
            subprocess.run(["git", "push", "origin", CFG["BRANCH"]], cwd=repo_dir, env=env, check=True)
        print(f"Pushed backup: https://github.com/{CFG['GITHUB_OWNER']}/{CFG['REPO_NAME']}/tree/{CFG['BRANCH']}/experiments/kaggle_runs/{run_id}")
    finally:
        try:
            askpass.unlink()
        except FileNotFoundError:
            pass


From https://github.com/TranTruongMMCII/UIT.CS2309
 * branch            feature/upr-cre -> FETCH_HEAD


Already up to date.
[feature/upr-cre 12d3535] exp: add SYSU Step 9 backup sysu_short_sysu_short_20260629_130838_5cb2e45
 10 files changed, 440 insertions(+)
 create mode 100644 experiments/kaggle_runs/sysu_short_sysu_short_20260629_130838_5cb2e45/README.md
 create mode 100644 experiments/kaggle_runs/sysu_short_sysu_short_20260629_130838_5cb2e45/logs/baseline_sysu_p2s15_sysu_short_sysu_short_20260629_130838_5cb2e45_log.txt
 create mode 100644 experiments/kaggle_runs/sysu_short_sysu_short_20260629_130838_5cb2e45/logs/upr_v02_sysu_p2s15_sysu_short_sysu_short_20260629_130838_5cb2e45_log.txt
 create mode 100644 experiments/kaggle_runs/sysu_short_sysu_short_20260629_130838_5cb2e45/manifest.json
 create mode 100644 experiments/kaggle_runs/sysu_short_sysu_short_20260629_130838_5cb2e45/relation_stats/baseline_sysu_p2s15_sysu_short_sysu_short_20260629_130838_5cb2e45_relation_stats_summary.csv
 create mode 100644 experiments/kaggle_runs/sysu_short_sysu_short_20260629_130838_5cb2e45/relation_stats

To https://github.com/TranTruongMMCII/UIT.CS2309.git
   5cb2e45..12d3535  feature/upr-cre -> feature/upr-cre
